In [0]:
from datetime import datetime
import pyspark.sql.functions as f
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
dbutils.widgets.text("run_date", "2026-11-04")
run_date = dbutils.widgets.get("run_date")
DROP_DATE = datetime.strptime(run_date, "%Y-%m-%d").date()

DAILY_PATH = (
    f"/Volumes/claim-etl/bronze/landing/healthcare/"
    f"daily/drop_date={DROP_DATE}/"
)

datasets = ["claims", "members", "providers"]
CLAIMS_PATH = DAILY_PATH + "claims/"
MEMBERS_PATH = DAILY_PATH + "members/"
PROVIDERS_PATH = DAILY_PATH + "providers/"

# CLAIMS SCHEMA
claims_schema = StructType([
    StructField("claim_id", StringType(), True),
    StructField("member_id", StringType(), True),
    StructField("provider_id", StringType(), True),
    StructField("service_date", StringType(), True),
    StructField("diagnosis_code", StringType(), True),
    StructField("procedure_code", StringType(), True),
    StructField("billed_amount", StringType(), True),
    StructField("paid_amount", StringType(), True),
    StructField("claim_status", StringType(), True),
    StructField("updated_at", StringType(), True)
])


# MEMBERS SCHEMA
members_schema = StructType([
    StructField("member_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("state", StringType(), True),
    StructField("city", StringType(), True),
    StructField("plan_type", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("operation", StringType(), True)
])


# PROVIDERS SCHEMA
providers_schema = StructType([
    StructField("provider_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("specialty", StringType(), True),
    StructField("state", StringType(), True),
    StructField("operation", StringType(), True)
])

INGEST_CONFIG = {
    "claims": {
        "schema": claims_schema,
        "source_path": f"{DAILY_PATH}claims/",
        "table_name": "`claim-etl`.bronze.claims"
    },
    "members": {
        "schema": members_schema,
        "source_path": f"{DAILY_PATH}members/",
        "table_name": "`claim-etl`.bronze.members"
    },
    "providers": {
        "schema": providers_schema,
        "source_path": f"{DAILY_PATH}providers/",
        "table_name": "`claim-etl`.bronze.providers"
    }
}

In [0]:
def ingest_to_bronze(dataset_name):

    dataset_name = dataset_name.strip().lower()

    config = INGEST_CONFIG.get(dataset_name)

    if config is None:
        raise ValueError(f"Invalid dataset: {dataset_name}")
    source_files = [
        file for file in dbutils.fs.ls(config["source_path"])
        if file.name.lower().endswith(".csv")
    ]

    if not source_files:
        raise ValueError(
            f"No CSV files found for {dataset_name}"
        )


    print(f"Loading {dataset_name} from {config['source_path']}")
    
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .schema(config["schema"])
        .load(config["source_path"])
    )

    df = df.withColumns({
        "_source_file": f.col("_metadata.file_path"),
        "_load_date": f.current_date(),
        "_batch_date": f.lit(DROP_DATE).cast("date")
    })

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"_batch_date = DATE '{DROP_DATE}'"
        )
        .partitionBy("_batch_date")
        .saveAsTable(config["table_name"])
    )

    return df

In [0]:
bronze_dfs = {}

for dataset in datasets:
    bronze_dfs[dataset] = ingest_to_bronze(dataset)

In [0]:
display(spark.table("`claim-etl`.bronze.claims").limit(10))
# spark.sql("DROP TABLE IF EXISTS `claim-etl`.bronze.claims")
display(spark.table("`claim-etl`.bronze.members").limit(10))
display(spark.table("`claim-etl`.bronze.providers").limit(10))

In [0]:
%sql
SELECT
    'claims' AS dataset,
    _batch_date,
    COUNT(*) AS record_count
FROM `claim-etl`.bronze.claims
GROUP BY _batch_date

UNION ALL

SELECT
    'members' AS dataset,
    _batch_date,
    COUNT(*) AS record_count
FROM `claim-etl`.bronze.members
GROUP BY _batch_date

UNION ALL

SELECT
    'providers' AS dataset,
    _batch_date,
    COUNT(*) AS record_count
FROM `claim-etl`.bronze.providers
GROUP BY _batch_date

ORDER BY dataset, _batch_date;